# Chapter 2: Thermodynamic Foundations — EOS and Flash Calculations

This notebook covers the thermodynamic building blocks used in production optimization:

- **Equations of State (EOS)**: Comparing SRK and Peng-Robinson models
- **Flash calculations**: TP-flash for phase equilibrium
- **Thermophysical properties**: Density, heat capacity, enthalpy, viscosity, and Joule-Thomson coefficient

Understanding these fundamentals is essential for accurate modeling of fluid behavior
across the production system — from reservoir to export.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


In [2]:
import matplotlib.pyplot as plt
import numpy as np

from neqsim import jneqsim

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

## 2.1 Helper: Create Fluid with a Given EOS

We define the same natural gas composition and create it with both SRK and PR
equations of state for comparison.

In [3]:
def create_fluid(eos_class, T_K, P_bara):
    """Create a natural gas fluid with the given EOS class."""
    fluid = eos_class(T_K, P_bara)
    fluid.addComponent("methane", 0.85)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.04)
    fluid.addComponent("n-butane", 0.02)
    fluid.addComponent("CO2", 0.015)
    fluid.addComponent("nitrogen", 0.005)
    fluid.setMixingRule("classic")
    return fluid

# Verify both EOS work
for eos_name, eos_cls in [("SRK", SystemSrkEos), ("PR", SystemPrEos)]:
    f = create_fluid(eos_cls, 273.15 + 25.0, 100.0)
    ops = ThermodynamicOperations(f)
    ops.TPflash()
    f.initProperties()
    print(f"{eos_name}: density = {f.getDensity('kg/m3'):.2f} kg/m³, "
          f"Z = {f.getPhase('gas').getZ():.4f}")

SRK: density = 101.25 kg/m³, Z = 0.7804
PR: density = 102.90 kg/m³, Z = 0.7422


## 2.2 Figure 1 — Density Comparison: SRK vs PR

Both cubic EOS give similar trends, but PR typically predicts slightly higher
liquid densities. For gas at moderate pressures, the difference is small.

In [4]:
pressures = np.linspace(10, 200, 25)
T_K = 273.15 + 25.0

density_srk = []
density_pr = []

for P in pressures:
    for eos_cls, result_list in [(SystemSrkEos, density_srk), (SystemPrEos, density_pr)]:
        fluid = create_fluid(eos_cls, T_K, float(P))
        ops = ThermodynamicOperations(fluid)
        ops.TPflash()
        fluid.initProperties()
        result_list.append(fluid.getDensity("kg/m3"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pressures, density_srk, 'b-o', markersize=3, linewidth=2, label='SRK EOS')
ax.plot(pressures, density_pr, 'r-s', markersize=3, linewidth=2, label='PR EOS')
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Gas Density (kg/m³)', fontsize=12)
ax.set_title('Figure 2.1: Gas Density — SRK vs Peng-Robinson at 25°C', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 210)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../figures/fig01_density_srk_vs_pr.png', dpi=150, bbox_inches='tight')
plt.show()

max_diff = max(abs(s - p) for s, p in zip(density_srk, density_pr))
print(f"Maximum density difference (SRK - PR): {max_diff:.2f} kg/m³")

Maximum density difference (SRK - PR): 1.77 kg/m³


C:\Users\ESOL\AppData\Local\Temp\ipykernel_38536\3359844916.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.3 Figure 2 — Heat Capacity (Cp) vs Temperature

The isobaric heat capacity Cp increases as temperature rises, especially near
the phase boundary where latent heat effects become significant.

In [5]:
temperatures_C = np.linspace(-40, 120, 30)
P_bara = 50.0
cp_values = []

for T_C in temperatures_C:
    T_K = 273.15 + float(T_C)
    fluid = create_fluid(SystemSrkEos, T_K, P_bara)
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()
    # Get Cp of the gas phase in J/(mol·K)
    cp = fluid.getPhase("gas").getCp("J/molK")
    cp_values.append(cp)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(temperatures_C, cp_values, 'g-^', markersize=4, linewidth=2)
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Cp (J/mol·K)', fontsize=12)
ax.set_title('Figure 2.2: Gas Heat Capacity vs Temperature at 50 bara', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig02_cp_vs_temperature.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Cp range: {min(cp_values):.2f} – {max(cp_values):.2f} J/(mol·K)")

Cp range: 47.79 – 59.44 J/(mol·K)


C:\Users\ESOL\AppData\Local\Temp\ipykernel_38536\2675627088.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.4 Figure 3 — Enthalpy vs Temperature

Tracking enthalpy across a temperature range at constant pressure reveals the
energy content of the gas. This is crucial for heat exchanger and cooler design.

In [6]:
enthalpy_values = []

for T_C in temperatures_C:
    T_K = 273.15 + float(T_C)
    fluid = create_fluid(SystemSrkEos, T_K, P_bara)
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()
    h = fluid.getEnthalpy("J/mol")
    enthalpy_values.append(h)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(temperatures_C, enthalpy_values, 'm-D', markersize=3, linewidth=2)
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Enthalpy (J/mol)', fontsize=12)
ax.set_title('Figure 2.3: Mixture Enthalpy vs Temperature at 50 bara', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig03_enthalpy_vs_temperature.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_38536\3941489549.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.5 Figure 4 — Gas Viscosity vs Pressure

Gas viscosity increases with pressure as molecular interactions intensify.
Accurate viscosity prediction is essential for pipeline hydraulic calculations.

In [7]:
pressures_visc = np.linspace(10, 200, 25)
viscosities = []
T_K_visc = 273.15 + 25.0

for P in pressures_visc:
    fluid = create_fluid(SystemSrkEos, T_K_visc, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()
    mu = fluid.getPhase("gas").getViscosity("kg/msec")
    viscosities.append(mu * 1e6)  # convert to µPa·s (microPascal-seconds)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pressures_visc, viscosities, 'c-o', markersize=3, linewidth=2)
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Gas Viscosity (µPa·s)', fontsize=12)
ax.set_title('Figure 2.4: Gas Viscosity vs Pressure at 25°C', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 210)
plt.tight_layout()
plt.savefig('../figures/fig04_viscosity_vs_pressure.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Viscosity range: {min(viscosities):.1f} – {max(viscosities):.1f} µPa·s")

Viscosity range: 11.2 – 22.8 µPa·s


C:\Users\ESOL\AppData\Local\Temp\ipykernel_38536\2374841098.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.6 Figure 5 — Joule-Thomson Coefficient vs Pressure

The Joule-Thomson (JT) coefficient describes temperature change during
isenthalpic expansion (e.g., through a choke valve). A positive JT coefficient
means cooling on expansion — important for hydrate risk assessment.

In [8]:
jt_coefficients = []

for P in pressures_visc:
    fluid = create_fluid(SystemSrkEos, T_K_visc, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()
    jt = fluid.getPhase("gas").getJouleThomsonCoefficient("C/bar")
    jt_coefficients.append(jt)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pressures_visc, jt_coefficients, 'k-v', markersize=4, linewidth=2)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='JT = 0 (inversion)')
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Joule-Thomson Coefficient (°C/bar)', fontsize=12)
ax.set_title('Figure 2.5: JT Coefficient vs Pressure at 25°C', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 210)
plt.tight_layout()
plt.savefig('../figures/fig05_jt_coefficient_vs_pressure.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_38536\7188782.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

Key takeaways from this chapter:

1. **SRK vs PR**: Both EOS give similar gas-phase predictions; PR tends to predict slightly different densities
2. **Heat capacity**: Cp increases with temperature and is pressure-dependent — important for heater/cooler sizing
3. **Enthalpy**: Monotonically increases with temperature; the slope reflects Cp
4. **Viscosity**: Gas viscosity increases with pressure, affecting pipeline pressure drop calculations
5. **Joule-Thomson**: Positive JT coefficient means cooling on expansion — critical for choke valve design and hydrate prevention

Next chapter: **Fluid Characterization and PVT Modeling** for reservoir fluids with heavy fractions.